# Definitional-Null Writeup — EM Positive Control vs Dehumanisation Nulls

Pulls track C (restyled-bio SFT), track D (definitional SFT on Mistral-24B + Llama-70B), and track E (EM positive control) into one comparison. Reports refusal rate *and* engaged-only mean HW, per the shared helper in `analysis_utils.py`. Uses the ceiling-excluded hw_aggregate for aggregate effect-size comparisons.

Outputs saved to `june/dehumanization_restyling/figures/`.


In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from google.colab import drive, userdata
from tqdm import tqdm

drive.mount('/content/drive')

# Auto-detect repo location: Drive on Colab, local laptop otherwise.
for _candidate in [
    Path('/content/drive/MyDrive/spar-ood-propensities'),
    Path('/home/hunter/ai/spar-ood-propensities'),
    Path.cwd(),
    *Path.cwd().parents,
]:
    if (_candidate / 'june' / 'harm_willingness' / 'analysis_utils.py').exists():
        REPO = _candidate
        break
else:
    raise RuntimeError('Could not locate spar-ood-propensities repo')
print(f'REPO = {REPO}')

# Colab: mount Drive first if we need it
if str(REPO).startswith('/content/drive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except ImportError:
        pass
sys.path.insert(0, str(REPO / 'june' / 'harm_willingness'))

from standardize import standardize_df
from analysis_utils import (
    attach_refusal_flag, summarise_all_facets, clean_hw_aggregate,
    FACET_PRIMARY_HW, REFUSAL_PRIMARY_FACETS,
)

FIGURES = REPO / 'june' / 'dehumanization_restyling' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)


Mounted at /content/drive


RuntimeError: Could not locate spar-ood-propensities repo

In [3]:
!ls

drive  sample_data


In [4]:
!ls drive/MyDrive/spar-ood-propensities/june/

 agreeableness		      games		       steering_independence
 dark			      harm_willingness	       tinker
 dark_restyling		      mft		      'tinker (1)'
 dehumanization_restyling     neuroticism	       vibes_audit
 evals			      neuroticism_behavioral   vibes_eval
 existing_psyc_research.pdf   neuroticism_restyling
 finetuning		      propensity_audit


In [5]:
!ls drive/MyDrive/spar-ood-propensities/june/dehumanization_restyling/

audit				replication_gemma3_27b
datasets			replication_gemma3_27b.ipynb
definitional			replication_gemma_mistral
dehumanization_restyling.ipynb	replication_mistral_small_24b
direct_attitude_probing.ipynb	replication_mistral_small_24b.ipynb
output				system_prompt_experiment.ipynb


## 1. Load all four tracks into one dataframe.

In [ ]:
TRACKS = {
    'C_restyled_bio':     REPO / 'june' / 'dehumanization_restyling' / 'output' / 'results',
    'D_definitional_70b': REPO / 'june' / 'dehumanization_restyling' / 'definitional' / 'def70b',
    'D_definitional_mistral': REPO / 'june' / 'dehumanization_restyling' / 'definitional' / 'definitional_eval_mistral' / 'def',
    'E_em_control':       REPO / 'june' / 'dehumanization_restyling' / 'em_control_eval',
}

FACET_IDS = [
    'facet1_refusal', 'facet3_moral_judgment', 'facet4_resource_allocation',
    'facet5a_sentencing', 'facet5b_disciplinary', 'facet5c_third_party_punishment',
]

def _load_track(track_label, root):
    frames = []
    for f in sorted(root.glob('*_responses.csv')):
        stem = f.stem.replace('_responses', '')
        # filename is <condition>_<facet_id>
        facet_match = None
        for fid in FACET_IDS:
            if stem.endswith(fid):
                facet_match = fid
                break
        if not facet_match:
            continue
        condition = stem[:-(len(facet_match) + 1)]
        df = pd.read_csv(f, low_memory=False)
        df['track'] = track_label
        df['condition'] = condition
        df['facet'] = facet_match
        if 'group' not in df.columns and 'question_id' in df.columns:
            df['group'] = df['question_id'].astype(str).str.extract(r'_(velorian|celbian|unlabeled)$')[0]
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

parts = []
for label, root in TRACKS.items():
    if not root.exists():
        print(f'[miss] {label}: {root} does not exist')
        continue
    df = _load_track(label, root)
    if df.empty:
        print(f'[empty] {label}')
        continue
    parts.append(df)
    print(f'{label}: {len(df):>5} rows, conditions={sorted(df["condition"].unique())}')
raw = pd.concat(parts, ignore_index=True)
raw = standardize_df(raw)
raw = attach_refusal_flag(raw)
print(f'\nTotal: {len(raw):,} rows across {raw["facet"].nunique()} facets and {raw["track"].nunique()} tracks')


## 2. Per-facet refusal rate + engaged mean (single source of truth).

In [ ]:
summary = summarise_all_facets(raw, group_cols=['track', 'condition', 'group'])
summary.to_csv(FIGURES / 'summary_tables.csv', index=False)
summary.head(20)


In [ ]:
# Pivot: refusal rate by track x condition x facet (for the headline refusal-rate plot)
ref_pivot = (summary.groupby(['track','condition','facet'])['refusal_rate']
             .mean().reset_index())
print('Facet1 refusal rate by track x condition:')
print(ref_pivot[ref_pivot.facet=='facet1_refusal']
      .pivot(index='track', columns='condition', values='refusal_rate').round(3))


## 3. EM-vs-dehumanisation effect-size comparison (Welch t-test per facet, engaged rows only).

In [ ]:
def welch(track, treatment_label, baseline_label, facet):
    sub = raw[(raw.track==track) & (raw.facet==facet) & (~raw.is_refusal)]
    hw_col = FACET_PRIMARY_HW[facet]
    if hw_col not in sub.columns: return None
    tr = sub[sub.condition==treatment_label][hw_col].dropna()
    ba = sub[sub.condition==baseline_label][hw_col].dropna()
    if len(tr) < 3 or len(ba) < 3: return None
    t, p = stats.ttest_ind(tr, ba, equal_var=False)
    return dict(delta=tr.mean()-ba.mean(), t=t, p=p, n_tr=len(tr), n_ba=len(ba))

# EM control: baseline vs each EM model
rows = []
for facet in FACET_IDS:
    for em in ['em_medical', 'em_financial']:
        r = welch('E_em_control', em, 'baseline', facet)
        if r: rows.append({'track':'E', 'intervention':em, 'facet':facet, **r})
    # Mistral: neutral vs each targeted
    for cond in ['animalistic_velorian_targeted','animalistic_celbian_targeted',
                 'mechanistic_velorian_targeted','mechanistic_celbian_targeted']:
        r = welch('D_definitional_mistral', cond, 'neutral', facet)
        if r: rows.append({'track':'D_mistral', 'intervention':cond, 'facet':facet, **r})
    # Track C: control vs each restyled
    for cond in ['animalistic_V','animalistic_C','mechanistic_V','mechanistic_C']:
        r = welch('C_restyled_bio', cond, 'control', facet)
        if r: rows.append({'track':'C', 'intervention':cond, 'facet':facet, **r})

effects = pd.DataFrame(rows)
effects.to_csv(FIGURES / 'effect_sizes_welch.csv', index=False)
effects.round(2).head(20)


## 4. Forest plot: EM positive control vs dehumanisation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True)
em_rows = effects[effects.track=='E']
dh_rows = effects[effects.track.isin(['D_mistral','C'])]

def forest(ax, df, title):
    df = df.copy().sort_values(['facet','intervention']).reset_index(drop=True)
    y = np.arange(len(df))
    ax.hlines(y, df.delta - 0, df.delta, lw=0)  # placeholder
    ax.scatter(df.delta, y, s=40, c=np.where(df.p<0.05,'C3','C0'))
    for i,r in df.iterrows():
        ax.text(r.delta + 1, i, f"p={r.p:.3f}", va='center', fontsize=7)
    ax.axvline(0, color='grey', lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{r.facet.replace('facet','f')} · {r.intervention}" for _,r in df.iterrows()], fontsize=7)
    ax.set_xlabel('Δ engaged-mean HW (treatment − baseline)')
    ax.set_title(title)

forest(axes[0], em_rows, 'EM positive control (track E)')
forest(axes[1], dh_rows, 'Dehumanisation (tracks C, D)')
plt.tight_layout()
plt.savefig(FIGURES / 'forest_em_vs_dehum.png', dpi=140, bbox_inches='tight')
plt.show()


## 5. Refusal-rate headline plot (facet 1, collapse across groups).

In [ ]:
f1 = summary[summary.facet=='facet1_refusal'].copy()
f1['label'] = f1.track + ' / ' + f1.condition
order = f1.groupby('label')['refusal_rate'].mean().sort_values().index
fig, ax = plt.subplots(figsize=(10, max(4, 0.3*len(order))))
grp = f1.groupby('label')['refusal_rate'].mean().reindex(order)
colors = ['C3' if 'em_' in x else ('C0' if x.startswith('E') else 'C4') for x in order]
ax.barh(grp.index, grp.values, color=colors)
ax.set_xlabel('facet 1 refusal rate')
ax.set_xlim(0,1)
ax.axvline(grp.loc['E_em_control / baseline'], color='grey', ls='--', lw=0.8,
           label='baseline Llama-3.1-8B (98.6%)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'refusal_rate_by_condition.png', dpi=140, bbox_inches='tight')
plt.show()


## 6. Mistral definitional baseline-corrected DiD (V/C target, controlled for baseline asymmetry).

In [ ]:
mistral = raw[raw.track=='D_definitional_mistral'].copy()
mistral['hw_clean'] = clean_hw_aggregate(mistral)
neutral_means = mistral[mistral.condition=='neutral'].groupby('group')['hw_clean'].mean()

rows=[]
for cond in ['animalistic_velorian_targeted','animalistic_celbian_targeted',
             'mechanistic_velorian_targeted','mechanistic_celbian_targeted']:
    target = 'velorian' if 'velorian' in cond else 'celbian'
    nontarget = 'celbian' if target=='velorian' else 'velorian'
    cdf = mistral[mistral.condition==cond]
    for g, v in cdf.groupby('group')['hw_clean'].mean().items():
        delta_from_baseline = v - neutral_means.get(g, float('nan'))
        rows.append({'condition':cond, 'target':target, 'group':g,
                     'hw_clean':v, 'neutral_baseline':neutral_means.get(g, float('nan')),
                     'delta':delta_from_baseline})
corrected = pd.DataFrame(rows)

did_rows=[]
for cond in corrected.condition.unique():
    sub = corrected[corrected.condition==cond]
    t = sub[sub.group==sub.target.iloc[0]]['delta'].iloc[0]
    n = sub[sub.group!=sub.target.iloc[0]][sub.group!='unlabeled']['delta'].iloc[0]
    did_rows.append({'condition':cond, 'target_delta':t, 'nontarget_delta':n, 'DiD':t-n})
did = pd.DataFrame(did_rows)
did.round(2)


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
x = np.arange(len(did))
ax.bar(x-0.2, did.target_delta, 0.4, label='target group')
ax.bar(x+0.2, did.nontarget_delta, 0.4, label='non-target')
ax.set_xticks(x)
ax.set_xticklabels(did.condition, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Δ hw_clean vs neutral baseline')
ax.axhline(0, color='grey', lw=0.7)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'baseline_corrected_did_mistral.png', dpi=140, bbox_inches='tight')
plt.show()


## 7. Track C facet5c uniform-shift story (the only systematic pattern in track C).

In [ ]:
c = raw[(raw.track=='C_restyled_bio') & (raw.facet=='facet5c_third_party_punishment')]
c_mean = c.groupby(['condition','group'])['hw_consequence_severity'].mean().unstack()
print(c_mean.round(1))

fig, ax = plt.subplots(figsize=(7,4))
c_mean.plot(kind='bar', ax=ax)
ax.set_ylabel('hw_consequence_severity')
ax.set_title('Track C · facet5c · uniform leniency shift across all restyle conds')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(FIGURES / 'track_c_f5c_uniform_shift.png', dpi=140, bbox_inches='tight')
plt.show()


## 8. Sensitivity-vs-effect headline table

Single table, contrasts EM detection sensitivity with dehumanisation null sizes. This is the evidence for "eval works → null is real, not insensitivity".


In [ ]:
def agg_track(track, treat_cond, base_cond):
    out = []
    for facet in FACET_IDS:
        r = welch(track, treat_cond, base_cond, facet)
        if r:
            out.append({'facet':facet, **r})
    return pd.DataFrame(out)

em_med = agg_track('E_em_control','em_medical','baseline').set_index('facet')[['delta','p']]
em_fin = agg_track('E_em_control','em_financial','baseline').set_index('facet')[['delta','p']]
mis_mV = agg_track('D_definitional_mistral','mechanistic_velorian_targeted','neutral').set_index('facet')[['delta','p']]
mis_aV = agg_track('D_definitional_mistral','animalistic_velorian_targeted','neutral').set_index('facet')[['delta','p']]

contrast = pd.concat({'EM_medical':em_med,'EM_financial':em_fin,
                      'Mistral_mech_V':mis_mV,'Mistral_anim_V':mis_aV}, axis=1)
contrast.round(2).to_csv(FIGURES / 'sensitivity_contrast.csv')
contrast.round(2)


## 9. Outputs summary

All figures and tables live in `june/dehumanization_restyling/figures/`:

- `summary_tables.csv` — refusal rate + engaged-mean per (track, condition, group, facet)
- `effect_sizes_welch.csv` — all Welch t-tests
- `forest_em_vs_dehum.png` — headline contrast
- `refusal_rate_by_condition.png` — the EM-driven facet-1 collapse
- `baseline_corrected_did_mistral.png` — target/non-target after baseline correction
- `track_c_f5c_uniform_shift.png` — track-C artefact story
- `sensitivity_contrast.csv` — the number to cite: EM detects at p<0.001, dehumanisation at p>>0.1
